# Machine Learning A9

## What we did
The goal of this phase is to develop a Deep Learning model capable of "lifting" 2D coordinates ($x, y$) extracted from MediaPipe into 3D coordinates ($z$) .Using the given Kinect data, we transformed it into MediaPipe-equivalent values. This ensures that the model trained on high-accuracy ground truth will be directly compatible with the real-time MediaPipe landmarks used in the final application.

## Evaluation Strategy
To further validate we use a validation strategy similar to previous to prove model accuracy and stability aswell as a Seed for reproducibility:

### 1. Physical Error Metric (MAE in cm)
While the model optimizes for Mean Squared Error (MSE) during training, we judge the final "Champion" based on the Grand Average MAE from the independent test set. This provides a human-readable metric, the average distance (in centimeters) the model is off across all joints. By using the test set for this metric, we ensure our "best" model is the one that generalizes most effectively to totally new movements.

### 2. Test Set Evaluation
We hold out **10 random sequences** entirely from the training and validation process. 
* **Grand Average:** We calculate the error across all joints and all frames in these sequences.
* **Per-Joint Analysis:** We generate a detailed report for all 13 joints (nose, wrists, ankles, etc.) to identify if the model has "weak links" in specific biomechanical movements.

### 3. Statistical Challenger Framework
Since our data consists of sequences, we do not utilize a standard K-Fold. Instead, we generate a statistical population by treating each of the 10 independent test sequences as an individual evaluation fold. This allows us to perform a Corrected Resampled t-Test to compare new experimental variants against the current "Champion" model.
* **Metric for Comparison:**  We specifically compare the Grand_Avg_Test_MAE_cm of the Challenger against the @dev model.
* **Probability Density Functions (PDF):** We visualize the error distributions of models to compare their stability.
* **p-value Validation:** A model is only promoted to the `@dev` alias if the improvement is statistically significant ($p < 0.05$), ensuring we don't promote models that simply got "lucky" on a specific data split.

## Experiment Tracking & Lifecycle
All experiments are logged to **DagsHub** via **MLflow**. This infrastructure allows us to:
1. **Compare Variants:** Side-by-side analysis of different architectures and hyperparameters.
2. **Model Registry:** Automatic versioning using aliases (`@prod`, `@dev`, `@backup`) to maintain a clean deployment pipeline.
3. **Artifact Traceability:** Every run stores its own error plots, per-joint CSV reports, and the trained `.pth` model file for full reproducibility.

## Data Representation

In this framework, the model utilizes **Sequences** to understand the physics of movement.

### The 30-Frame Window
Each sequence consists of **30 consecutive frames**, representing approximately **1 second of real-time movement** (at 30 FPS). 
* **Input:** A tensor of shape `(30, 26)`, representing the ($x, y$) coordinates of 13 joints over 1 second.
* **Objective:** Predict the corresponding depth ($z$) for those same 30 frames.

By using sequences units can observe the movement of limbs.


### Robust "Blind" Testing
To ensure the model is truly learning biomechanics rather than memorizing specific files, we implement a strict **Hold-out Test Set**:
1. **The Pool:** All available data from various Kinect recordings are broken into thousands of 1-second segments.
2. **The Extraction:** 10 specific sequences are randomly sampled from this global pool and locked away.
3. **Diversity:** Because these 10 segments are chosen at random, they typically represent a cross-section of different users, body types, and movement speeds.
4. **Independence:** These 10 sequences are never seen by the model during the training or validation phases. 

The resulting **Grand Average Error** is therefore a reliable indicator of how the model will perform on entirely new, unseen human subjects in a real-world environment.

## Further validation
We also do another validation were we test some random sequence from all data and check that the output is in a correct format. This is because we actually dont use this yet in the project and can validate that way that it produces the correct data. Here is a sample output of the validation script:
```markdown
======================================================================
Joint Name         | Pred Z (m)   | True Z (m)   | Diff (cm) 
----------------------------------------------------------------------
nose               |     0.0369   |     0.0363   |     0.07 cm
left_shoulder      |    -0.1167   |    -0.1175   |     0.08 cm
left_elbow         |    -0.0129   |    -0.0115   |     0.14 cm
right_shoulder     |    -0.0062   |     0.0032   |     0.94 cm
right_elbow        |     0.0135   |     0.0148   |     0.13 cm
left_wrist         |    -0.1859   |    -0.1924   |     0.65 cm
right_wrist        |     0.0847   |     0.0816   |     0.31 cm
left_hip           |     0.0942   |     0.0976   |     0.34 cm
right_hip          |    -0.0210   |    -0.0436   |     2.26 cm
left_knee          |     0.0129   |     0.0115   |     0.14 cm
right_knee         |     0.0605   |     0.0706   |     1.01 cm
left_ankle         |     0.0692   |     0.0625   |     0.67 cm
right_ankle        |    -0.1106   |    -0.1273   |     1.67 cm
----------------------------------------------------------------------
Average Frame Error: 0.65 cm
======================================================================
```

## Results from hyperparameter tuning
### Setup
To ensure validity between models we use 50 epochs with a patience of 5 to reduce the risk of overfitting using the validation MSE.

### Baselines
```python
config = {
    "variant": "GRU-Z-Lifting",
    "seq_length": 30,
    "batch_size": 32,
    "hidden_size": 128,
    "num_layers": 2,
    "learning_rate": 0.001,
    "rnn_type": 'GRU',

    "epochs": 50,
    "patience": 5,
    "seed": 42
}
```
#### Baseline LSTM
![img](img/baseline_plots.png)
[Link to experiment](https://dagshub.com/SamuelFredricBerg/4dt907/experiments#/experiment/m_c0c0456057704e2f84510bec7eea0484)

Grand average error in cm: 0.8886 cm
Std: 0.1283 cm

#### Baseline GRU
![img](img/lstmvsgru.png)
[Link to experiment](https://dagshub.com/SamuelFredricBerg/4dt907/experiments#/experiment/m_54fdd2a187ad494f89aa09656ed842aa)


Grand average error in cm: 1.059 cm
Std: 0.2533 cm

#### LSTM with 256 hidden
![img](img/256hidden.png)
![img](img/256hiddencomp.png)
